# Data Scientist Workflow - Step 1: Explore prod, build the Analytical Base Table

This is the **first thing a data scientist does** in the dev sandbox. They have
**read-only** access to production data (`ML_FRAUD_PRODUCTION`). Before building any
feature views, they explore the data and apply a **selective transform** into a clean,
labeled **analytical base table (ABT)** in their own dev sandbox.

Run as `ML_DEV_ROLE` - full control in dev, read-only on prod, and (deliberately)
no ability to write to production.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
session.sql("USE ROLE ML_DEV_ROLE").collect()
session.sql("USE WAREHOUSE CORTEX_CODE_WH").collect()
print("role:", session.get_current_role())

## 1. Explore production data (read-only)\n\nThe DS can read prod, but not write it.

In [ ]:
-- Class balance + date window of the production events
SELECT COUNT(*) AS n_txns,
       SUM(IS_LAUNDERING) AS n_laundering,
       ROUND(100.0*SUM(IS_LAUNDERING)/COUNT(*), 3) AS laundering_pct,
       MIN(EVENT_TS) AS first_ts, MAX(EVENT_TS) AS last_ts
FROM ML_FRAUD_PRODUCTION.CURATED.TXN_EVENTS

In [ ]:
-- Laundering rate by payment channel (why ACH matters in this data)
SELECT PAYMENT_FORMAT,
       COUNT(*) AS n,
       SUM(IS_LAUNDERING) AS fraud,
       ROUND(100.0*SUM(IS_LAUNDERING)/COUNT(*), 4) AS fraud_pct
FROM ML_FRAUD_PRODUCTION.CURATED.TXN_EVENTS
GROUP BY PAYMENT_FORMAT ORDER BY n DESC

## 2. Selective transform -> dev ABT

The canonical transform lives in `transforms/base_features.py` so the *same* logic
is promoted to prod later. Here the DS builds the **dev** ABT from prod data.

In [ ]:
import sys, os
# git-backed workspace: repo root is the working dir; make transforms importable
for p in (os.getcwd(), os.path.dirname(os.getcwd())):
    if p not in sys.path:
        sys.path.insert(0, p)
from transforms.base_features import build_abt

abt = build_abt(session, env="dev")
print("Built ABT:", abt)

In [ ]:
-- Inspect the ABT the feature views will be built on
SELECT SPLIT, COUNT(*) AS n, SUM(IS_LAUNDERING) AS positives
FROM ML_FRAUD_DEV_SANDBOX.CURATED.FRAUD_ABT
GROUP BY SPLIT ORDER BY MIN(EVENT_TS)

## Next
The ABT (`ML_FRAUD_DEV_SANDBOX.CURATED.FRAUD_ABT`) is now the single input for the
dev feature store and model training. Continue with the feature-store setup, then
train and track an experiment - all in the dev sandbox.